# Çürük modelini Colab'da eğit

Bu defter Google'ın ücretsiz GPU'sunda çürük tespit modelini eğitir ve sonunda `best.pt` dosyasını bilgisayarına indirir.

**Başlamadan önce:** üst menüden **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seç ve kaydet.

Sonra **Çalışma zamanı → Tümünü çalıştır** de. Toplam süre yaklaşık 1-2 saat. Sekmeyi kapatma; uzun süre dokunulmazsa Colab oturumu kapatabilir.

> Araştırma prototipidir, klinik karar için değildir.

## 1. GPU var mı?

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU yok: Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU seç, sonra yeniden çalıştır."
print("GPU:", torch.cuda.get_device_name(0))

## 2. Kodu ve kütüphaneleri indir

In [ ]:
import os, subprocess
REPO = "https://github.com/sahinemin751-cmyk/Tasarla-di-hekimi-yapay-zeka-asistan.git"
if not os.path.exists("/content/asistan"):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/asistan"], check=True)
%cd /content/asistan
if not os.path.exists("scripts/prepare_caries_dataset.py"):
    # PR henüz birleştirilmediyse prototip dalını kullan
    !git fetch --depth 1 origin caries-prototip && git checkout -q FETCH_HEAD
!pip install -q ultralytics

## 3. Veri setini indir ve hazırla (Zenodo, 1,6 GB, birkaç dakika)

In [ ]:
!wget -q --show-progress -O /content/Dataset.zip "https://zenodo.org/records/14827784/files/Dataset.zip?download=1"
!python scripts/prepare_caries_dataset.py --source /content/Dataset.zip --out /content/data/caries

Yukarıdaki çıktıda **YOLO etiketli** görüntü sayısı 6.000 civarında olmalı. Çok düşükse (ör. 0) burada dur ve çıktıyı Claude'a gönder.

## 4. Modeli eğit (en uzun adım, ~1-2 saat)

Daha hızlı denemek için `EPOCHS` değerini 20 yapabilirsin; sonuç biraz daha zayıf olur.

In [ ]:
EPOCHS = 60
!python scripts/train_caries.py --data /content/data/caries/data.yaml --epochs {EPOCHS} --device 0

## 5. Birkaç test görüntüsünde sonucu gör

In [ ]:
import glob
from ultralytics import YOLO
from IPython.display import Image, display
best = sorted(glob.glob("runs/caries*/weights/best.pt"))[-1]
model = YOLO(best)
for path in sorted(glob.glob("/content/data/caries/images/test/*"))[:6]:
    out = model.predict(path, conf=0.35, verbose=False)[0]
    out.save(filename="/content/tahmin.jpg")
    display(Image("/content/tahmin.jpg", width=480))

## 6. Modeli bilgisayarına indir

İnen `best.pt` dosyasını proje klasöründe `runs\caries\weights\` içine koy, sonra kamera demosunu kendi bilgisayarında çalıştır:

```
python scripts\camera_demo.py --weights runs\caries\weights\best.pt
```

Ekran kartı olmadan da çalışır; sadece kare hızı daha düşük olur.

In [ ]:
from google.colab import files
files.download(best)